# $G_a$ family: nonlinearity, Walsh spectrum, APN/permutation verification, and affine equivalence

This notebook studies the **first generalised Li-Kaleyski family**

$$G_a(x,y,z) = \bigl(x^{q+1}+ax^qz+yz^q,\; x^qz+y^{q+1},\; xy^q+ay^qz+z^{q+1}\bigr),\quad a\in\mathbb{F}_{2^m}^*,\quad q=2^i,\quad \gcd(i,m)=1,\quad m \text{ odd.}$$

**Governing polynomial** (Theorem 2.3 of the paper):
$$Q_a(T) = T^{q^2+q+1}+aT+1.$$
$G_a$ is a permutation $\iff$ $G_a$ is APN $\iff$ $Q_a$ has no root in $\mathbb{F}_{2^m}$.

**Diagonal equivalence** (Proposition 2.6): $G_1 = A_1\circ G_a\circ A_2$ for diagonal $\mathbb{F}_{2^m}$-linear bijections $A_1,A_2$ if and only if $a^{q^2+q+1}=1$. The set of such $a$ is a subgroup of $\mathbb{F}_{2^m}^*$ of order $d_0=\gcd(q^2+q+1,2^m-1)$.

**Cells:**
1. Exact nonlinearity of $G_1$
2. Walsh-spectrum comparison across all good $a$ (necessary condition for affine equivalence)
3. Permutation / APN / root-condition verification table
4. Diagonal equivalence check ($a^{q^2+q+1}=1$, with explicit witnesses)
5. Full $\mathbb{F}_2$-linear affine equivalence check between $G_a$ and $G_1$

In [ ]:
from sage.all import *
from sage.crypto.boolean_function import BooleanFunction
import time

def compute_nl_G1(m, i):
    F  = GF(2**m, 'w')
    q  = 2**i
    print(f'--- Exact Nonlinearity of G_1  (m={m}, i={i}, q={q}) ---')
    bits = []
    for x in F:
        xq = x**q; xq1 = x*xq
        for y in F:
            yq = y**q; yq1 = y*yq
            for z in F:
                zq = z**q; zq1 = z*zq
                # G_1: a=1
                c1 = xq1 + xq*z + y*zq
                c2 = xq*z + yq1
                c3 = x*yq + yq*z + zq1
                bits.append(int((c1 + c2 + c3).trace()))
    bf = BooleanFunction(bits)
    nl = bf.nonlinearity()
    print(f'Exact Nonlinearity of G_1: {nl}')
    return nl

compute_nl_G1(m=7, i=1)


In [ ]:
from sage.all import *
from sage.crypto.boolean_function import BooleanFunction
import time

def get_invariants_Ga(m, i, a_val):
    F = GF(2**m, 'w'); q = 2**i
    bits = []
    for x in F:
        xq=x**q; xq1=x*xq
        for y in F:
            yq=y**q; yq1=y*yq
            for z in F:
                zq=z**q; zq1=z*zq
                c1 = xq1 + a_val*xq*z + y*zq
                c2 = xq*z + yq1
                c3 = x*yq + a_val*yq*z + zq1
                bits.append(int((c1+c2+c3).trace()))
    bf = BooleanFunction(bits)
    return bf.nonlinearity(), tuple(sorted(bf.walsh_hadamard_transform()))

def run_walsh_comparison_Ga(m, i):
    F = GF(2**m, 'w'); q = 2**i; d = q**2+q+1
    P.<T> = F[]
    print(f'--- Walsh-spectrum comparison for G_a  (m={m}, i={i}, q={q}) ---')
    print('Computing invariants for G_1 ...')
    t0 = time.time()
    nl1, spec1 = get_invariants_Ga(m, i, F(1))
    print(f'G_1: NL={nl1}  ({time.time()-t0:.1f}s)\n')
    print(f'{"a":<20} {"NL":<10} {"Spec=G_1?":<12} {"time":>6}')
    print('-'*52)
    for a in F:
        if a == 0 or a == 1: continue
        Qa = T**d + a*T + 1
        if Qa.roots(): continue   # skip bad a
        t0 = time.time()
        nla, speca = get_invariants_Ga(m, i, a)
        match = (nl1==nla and spec1==speca)
        tag = 'MATCH' if match else 'DIFFERENT'
        print(f'{str(a):<20} {nla:<10} {tag:<12} {time.time()-t0:>5.1f}s')

run_walsh_comparison_Ga(m=7, i=1)


In [ ]:
from sage.all import *
import time

def build_Ga_lut(m, i, a_val):
    F = GF(2**m, 'w'); q = 2**i
    lut = {}
    for x in F:
        xq=x**q; xq1=x*xq
        for y in F:
            yq=y**q; yq1=y*yq
            for z in F:
                zq=z**q; zq1=z*zq
                c1 = xq1 + a_val*xq*z + y*zq
                c2 = xq*z + yq1
                c3 = x*yq + a_val*yq*z + zq1
                lut[(x,y,z)] = (c1,c2,c3)
    return lut

def check_APN_Ga(m, i, a_val):
    F = GF(2**m, 'w'); lut = build_Ga_lut(m, i, a_val)
    triples = list(lut.keys())
    for (A,B,C) in triples:
        if A==B==C==F(0): continue
        cnt = sum(1 for (x,y,z) in triples
                  if (lut[(x+A,y+B,z+C)][0]+lut[(x,y,z)][0]==lut[(A,B,C)][0] and
                      lut[(x+A,y+B,z+C)][1]+lut[(x,y,z)][1]==lut[(A,B,C)][1] and
                      lut[(x+A,y+B,z+C)][2]+lut[(x,y,z)][2]==lut[(A,B,C)][2]))
        if cnt > 2: return False
    return True

def verify_table_Ga(m, i):
    F = GF(2**m, 'w'); q = 2**i; d = q**2+q+1
    P.<T> = F[]
    d0 = gcd(d, 2**m-1)
    print(f'm={m}, i={i}, q={q},  q^2+q+1={d},  2^m-1={2**m-1},  d0={d0}')
    print(f'Predicted diag-linear-equiv: a^(q^2+q+1)=1  ({d0} element(s))')
    print(f'Checking: FULL AFFINE  [N=2^{n}={2**n}]'.replace('{n}',str(3*m)).replace('{2**n}',str(2**(3*m))))
    print('='*72)
    print(f'G_1 reference built.')
    print()
    hdr = f'{"a":<20} {"Qa root-free?":<16} {"perm?":<8} {"APN?":<8} {"time":>6}'
    print(hdr); print('-'*62)
    for a in F:
        if a == 0: continue
        t0 = time.time()
        Qa = T**d + a*T + 1
        no_root = (len(Qa.roots()) == 0)
        lut = build_Ga_lut(m, i, a)
        is_perm = (len(set(lut.values())) == 2**(3*m))
        is_apn = check_APN_Ga(m, i, a) if m <= 5 else 'skipped'
        consistent = (no_root == is_perm == (is_apn if isinstance(is_apn, bool) else no_root))
        print(f'{str(a):<20} {str(no_root):<16} {str(is_perm):<8} '
              f'{str(is_apn):<8} {time.time()-t0:>5.1f}s   {"OK" if consistent else "MISMATCH!"}')

verify_table_Ga(m=3, i=1)
print()
verify_table_Ga(m=5, i=1)


In [ ]:
from sage.all import *

def check_diagonal_equiv_Ga(m, i):
    # G_1 = A1 o G_a o A2 for diagonal maps A1(u,v,w)=(mu*u,nu*v,rho*w),
    # A2(x,y,z)=(lam1*x,lam2*y,lam3*z).  Matching the 8 monomials gives
    # (Proposition 2.6, equations E1-E8):
    # E1: mu*lam1^{q+1}=1       E2: mu*a*lam1^q*lam3=1
    # E3: mu*lam2*lam3^q=1      E4: nu*lam1^q*lam3=1
    # E5: nu*lam2^{q+1}=1       E6: rho*lam1*lam2^q=1
    # E7: rho*a*lam2^q*lam3=1   E8: rho*lam3^{q+1}=1
    # Eliminating mu,nu,rho and deriving consistency yields a^{q^2+q+1}=1.
    F = GF(2**m, 'w'); q = 2**i; d = q**2+q+1; d0 = gcd(d, 2**m-1)
    print(f'=== Diagonal equivalence G_a ~ G_1  (m={m}, i={i}, q={q}) ===')
    print(f'Predicted condition: a^{d}=1,  subgroup order d0={d0}')
    nonzero = [e for e in F if e != 0]

    def G1():
        lut = {}
        for x in F:
            xq=x**q; xq1=x*xq
            for y in F:
                yq=y**q; yq1=y*yq
                for z in F:
                    zq=z**q; zq1=z*zq
                    lut[(x,y,z)]=(xq1+xq*z+y*zq, xq*z+yq1, x*yq+yq*z+zq1)
        return lut

    lut1 = G1()
    diag_equiv = []
    for a in F:
        if a == 0: continue
        pred = (a**d == F(1))
        found = False
        for lam2 in nonzero:
            # From Prop 2.6 sufficiency: r=a^{-q}, lam1=r*lam2, lam3=lam1/a
            r = a**(-q)
            lam1 = r * lam2
            lam3 = lam1 / a
            mu = lam1**(-(q+1)); nu = lam2**(-(q+1)); rho = lam3**(-(q+1))
            if not (mu*lam1**(q+1)==1 and mu*a*lam1**q*lam3==1 and
                    mu*lam2*lam3**q==1 and nu*lam1**q*lam3==1 and
                    nu*lam2**(q+1)==1 and rho*lam1*lam2**q==1 and
                    rho*a*lam2**q*lam3==1 and rho*lam3**(q+1)==1):
                continue
            Ga_lut = {}
            for x in F:
                xq=x**q; xq1=x*xq
                for y in F:
                    yq=y**q; yq1=y*yq
                    for z in F:
                        zq=z**q; zq1=z*zq
                        Ga_lut[(x,y,z)]=(xq1+a*xq*z+y*zq, xq*z+yq1, x*yq+a*yq*z+zq1)
            err = sum(1 for (x,y,z) in Ga_lut
                      if (mu*Ga_lut[(lam1*x,lam2*y,lam3*z)][0],
                          nu*Ga_lut[(lam1*x,lam2*y,lam3*z)][1],
                          rho*Ga_lut[(lam1*x,lam2*y,lam3*z)][2]) != lut1[(x,y,z)])
            if err == 0: found = True; break
        if found: diag_equiv.append(a)
        match = 'OK' if (pred==found) else 'MISMATCH!'
        print(f'  a={str(a):<20}  a^d=1: {str(pred):<6}  witness: {str(found):<6}  {match}')
    print(f'Diagonal-equiv elements ({len(diag_equiv)}): {[str(a) for a in diag_equiv]}')
    print()

check_diagonal_equiv_Ga(m=3, i=1)
check_diagonal_equiv_Ga(m=5, i=1)


In [ ]:
from sage.all import *
import time

# Gaussian elimination helpers for partial F_2-linear map
# Each insertion costs O(n) vs O(N^2) for naive XOR-closure.
def la(rows, d, i):
    rows = list(rows)
    for rd,ri in rows:
        if (d>>(rd.bit_length()-1))&1: d^=rd; i^=ri
    if d==0: return (i==0), rows
    rows.append((d,i)); return True, rows

def lq(rows, d):
    i=0
    for rd,ri in rows:
        if (d>>(rd.bit_length()-1))&1: d^=rd; i^=ri
    return i if d==0 else None

def lab(rows, items):
    for d,i in items.items():
        ok,rows = la(rows,d,i)
        if not ok: return False,None
    return True, rows

def leval(rows, NN):
    lut=[0]*NN
    for x in range(1,NN):
        hb=x.bit_length()-1; le=lq(rows,1<<hb)
        if le is None: return None
        lut[x]=lut[x^(1<<hb)]^le
    return lut

def check_linear_equiv(F_lut, G_lut, n):
    NN=1<<n; basis=[1<<k for k in range(n)]
    def is_lin(lt):
        mat=[lt[basis[k]] for k in range(n)]
        for x in range(NN):
            ex=0
            for k in range(n):
                if x&basis[k]: ex^=mat[k]
            if lt[x]!=ex: return False
        return True
    for v0 in range(1,NN):
        fv0=F_lut[v0]
        if fv0==0: continue
        ok,rows=la([],0,G_lut[0])
        if not ok: continue
        ok,rows=la(rows,fv0,G_lut[basis[0]])
        if not ok: continue
        sp=[(0,0),(basis[0],v0)]; good=True
        for j in range(1,n):
            outs={xo for _,xo in sp}; placed=False
            for w in range(1,NN):
                if w in outs: continue
                ne={}; cf=False
                for si,so in sp:
                    fx=F_lut[w^so]; gx=G_lut[basis[j]^si]
                    qv=lq(rows,fx)
                    if qv is not None:
                        if qv!=gx: cf=True; break
                    elif fx in ne:
                        if ne[fx]!=gx: cf=True; break
                    else: ne[fx]=gx
                if cf: continue
                ok2,r2=lab(rows,ne)
                if ok2:
                    rows=r2; sp=sp+[(basis[j]^si,w^so) for si,so in sp]
                    placed=True; break
            if not placed: good=False; break
        if not good: continue
        L2=[0]*NN
        for xi,xo in sp: L2[xi]=xo
        if len(set(L2))!=NN: continue
        L1=leval(rows,NN)
        if L1 is None: continue
        if not all(L1[F_lut[L2[x]]]==G_lut[x] for x in range(NN)): continue
        if not is_lin(L2): continue
        return True, L2, L1
    return False, None, None

def build_int_luts_Ga(m, i, a_sage, F):
    q=2**i; elts=list(F)
    idx={e:k for k,e in enumerate(elts)}
    def ti(x,y,z): return idx[x]|(idx[y]<<m)|(idx[z]<<(2*m))
    Ga={}; G1={}
    a=a_sage
    for x in elts:
        xq=x**q; xq1=x*xq
        for y in elts:
            yq=y**q; yq1=y*yq
            for z in elts:
                zq=z**q; zq1=z*zq; key=ti(x,y,z)
                Ga[key]=ti(xq1+a*xq*z+y*zq, xq*z+yq1, x*yq+a*yq*z+zq1)
                G1[key]=ti(xq1+xq*z+y*zq,   xq*z+yq1, x*yq+yq*z+zq1)
    return Ga, G1

def shifted(lut, c2, NN):
    gc2=lut[c2]
    return {x: lut[x^c2]^gc2 for x in range(NN)}

def check_affine_equiv_Ga(m, i, full_affine=True):
    F=GF(2**m,'w'); q=2**i; d=q**2+q+1; d0=gcd(d,2**m-1)
    NN=2**(3*m); n=3*m; P.<T>=F[]
    print(f'm={m}, i={i}, q={q},  q^2+q+1={d},  2^m-1={2**m-1},  d0={d0}')
    mode='FULL AFFINE' if full_affine else 'LINEAR ONLY'
    print(f'Predicted diag-linear-equiv: a^(q^2+q+1)=1  ({d0} element(s))')
    print(f'Checking: {mode}  [N=2^{n}={NN}]')
    print('='*72)
    t_build=time.time()
    _,G1d=build_int_luts_Ga(m,i,F(1),F)
    G1=[G1d[x] for x in range(NN)]
    print(f'G_1 lookup built in {time.time()-t_build:.2f}s\n')
    hdr=f'{"a":<20} {"a^d=1?":<10} {"perm?":<8} {"APN?":<6} {"affine~G_1?":<13} {"consistent?":<13} {"time":>6}'
    print(hdr); print('-'*80)
    for a_sage in F:
        if a_sage==0: continue
        t0=time.time()
        Qa=T**d+a_sage*T+1; no_root=(len(Qa.roots())==0)
        diag_eq=(a_sage**d==F(1))
        Gad,_=build_int_luts_Ga(m,i,a_sage,F)
        Ga=[Gad[x] for x in range(NN)]
        is_perm=(len(set(Ga))==NN)
        if full_affine:
            found=False
            for c2 in range(NN):
                Fc2=shifted(Ga,c2,NN)
                ok,_,_=check_linear_equiv(Fc2,G1,n)
                if ok: found=True; break
        else:
            ok,_,_=check_linear_equiv(Ga,G1,n); found=ok
        consistent=(no_root==is_perm==found)
        print(f'{str(a_sage):<20} {str(diag_eq):<10} {str(is_perm):<8} '
              f'{str(no_root):<6} {str(found):<13} {str(consistent):<13} {time.time()-t0:>5.1f}s')
    print('-'*80)

check_affine_equiv_Ga(m=3, i=1, full_affine=True)
print()
check_affine_equiv_Ga(m=5, i=1, full_affine=True)
